# 36. Valid Sudoku

[Problem](https://leetcode.com/problems/valid-sudoku/) · difficulty: medium

Two approaches that do the identical scan and differ only in what remembers the digits seen.
The judge says the bitmask is faster; measuring locally says the opposite. Both are right, and
the reason is worth more than the problem itself.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0036-valid-sudoku'
sys.path.insert(0, str(ROOT))

from lc.harness import load_module, load_solutions

solutions = load_solutions(PROBLEM)
tests = load_module(next(PROBLEM.glob('test_*.py')))
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Remembers digits in | LeetCode |
|---|---|---|---|---|
| `SolutionHashSet` | O(1) | O(1) | 27 sets | 2 ms |
| `SolutionBitmask` | O(1) | O(1) | 27 integers | 0 ms |

O(1) because the board is fixed at 9×9 — in board width `n`, O(n²) time for both.


## The box index

The only piece of arithmetic in either solution: `3 * (row // 3) + col // 3` flattens the grid
of 3×3 boxes into `0..8`. Integer division picks the band; the multiply gives each band three
slots.


In [ ]:
for row in range(9):
    print(' '.join(str(3 * (row // 3) + col // 3) for col in range(9)))


Each digit has to be unique within its row, its column, and its box — three constraint families
over the same 81 cells, which is why one pass with three lookups settles it.


## Seen-sets against seen-bits

`SolutionBitmask` stores the same information in the bits of an integer: digit `d` is bit
`d - 1`, so a row's state is one number instead of a set.


In [ ]:
row_values = ['5', '3', '.', '.', '7', '.', '.', '.', '.']

seen_set = set()
seen_bits = 0
for value in row_values:
    if value == '.':
        continue
    seen_set.add(value)
    seen_bits |= 1 << (ord(value) - ord('1'))

print(f'set    : {sorted(seen_set)}')
print(f'bits   : {seen_bits:>9} = 0b{seen_bits:09b}   (bit d-1 set for digit d)')
print(f'digit 3 seen? set={"3" in seen_set}  bits={bool(seen_bits & (1 << 2))}')
print(f'digit 4 seen? set={"4" in seen_set}  bits={bool(seen_bits & (1 << 3))}')


## Which is faster depends entirely on the board

Four shapes: a solved board (every cell filled, no early exit), the problem's example, an empty
board, and one that is invalid at the second cell.


In [ ]:
import timeit

SOLVED = [[str((3 * r + r // 3 + c) % 9 + 1) for c in range(9)] for r in range(9)]
assert all(len(set(row)) == 9 for row in SOLVED)

BOARDS = {
    'solved (81 clues)': SOLVED,
    'example (30 clues)': tests.VALID_BOARD,
    'empty': [['.'] * 9 for _ in range(9)],
    'invalid at (0,1)': tests.BOX_DUPLICATE_BOARD,
}

def micros(solution, board):
    call = min(timeit.repeat(lambda: solution().isValidSudoku([r[:] for r in board]),
                             number=2000, repeat=5))
    copy = min(timeit.repeat(lambda: [r[:] for r in board], number=2000, repeat=5))
    return (call - copy) / 2000 * 1e6

print(f"{'approach':<20}" + ''.join(f'{name:>22}' for name in BOARDS))
for solution in solutions:
    row = ''.join(f'{micros(solution, board):19.2f} us' for board in BOARDS.values())
    print(f'{solution.__name__:<20}{row}')


The bitmask loses where there is work to do and wins where there is not. That is the opposite of
what the technique promises — so where does the time go?


In [ ]:
MASKS = {str(d): 1 << (d - 1) for d in range(1, 10)}

pieces = {
    '81x  ord(c) - ord("1")': ('[ord(c) - 49 for c in s]', ''),
    '81x  dict lookup': ('[MASKS[c] for c in s]', ''),
    '81x  set.add': ('[t.add(c) for c in s]', ''),
}
setup = "s='123456789'*9; MASKS={str(d): 1 << (d-1) for d in range(1,10)}; t=set()"
for label, (stmt, _) in pieces.items():
    t = min(timeit.repeat(stmt, setup=setup, number=3000, repeat=7)) / 3000 * 1e6
    print(f'{label:<26}{t:6.2f} us')


There it is. Converting a character to a bit costs *more* than adding it to a set, because
`set.add` is a C function call while `1 << (ord(v) - ord("1"))` is three interpreted bytecodes.
A lookup table shaves a little off and still does not close the gap.

The bitmask's real advantages are elsewhere: 27 integers cost less to create than 27 sets (the
empty-board column), and an early rejection pays almost nothing (the invalid column).


## Why the judge disagrees

LeetCode reports 0 ms for the bitmask and 2 ms for the sets. Nothing above contradicts that —
its 507 test cases are mostly small, sparsely filled and invalid, which is precisely the regime
where the bitmask wins. And the whole spread being measured is about 3 µs, reported at
millisecond resolution.

A 0-versus-2 ms verdict on a 3 µs difference is not a measurement of the algorithm; it is a
measurement of the judge's test distribution.


## Takeaway

- In CPython, replacing a C-implemented builtin with hand-written arithmetic usually loses, even
  when the operation count drops. The bitmask is a clear win in C and a 24% loss here on a full
  board.
- Which input you measure decides the answer: the same pair of implementations ranks both ways
  depending on whether the board is full or empty, valid or rejected at cell two.
- The judge's millisecond figure is a coarse sample of one distribution. It is evidence, not a
  measurement — and it is worth reproducing locally before drawing a conclusion from it.
